In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

data = pd.read_csv('data/car_prices.csv')

print('Форма датасета:', data.shape)
data.head()

In [ ]:
data.describe()

In [ ]:
data.dtypes

In [ ]:
y = data['price']

# Только числовые признаки для первой части (обработка пропусков)
X = data.drop(['price'], axis=1).select_dtypes(exclude=['object'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

print('Размер обучающей выборки:', X_train.shape)
print('Размер тестовой выборки:', X_test.shape)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

def score_dataset(X_train, X_test, y_train, y_test):
    model = RandomForestRegressor(n_estimators=10, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return mean_absolute_error(y_test, preds)

In [ ]:
print('Размер тренировочной выборки:', X_train.shape)

missing_val_count = X_train.isnull().sum()
print('\nПропуски по столбцам:')
print(missing_val_count[missing_val_count > 0])

In [ ]:
cols_with_missing = [col for col in X_train.columns if X_train[col].isnull().any()]

reduced_X_train = X_train.drop(cols_with_missing, axis=1)
reduced_X_test = X_test.drop(cols_with_missing, axis=1)

print('MAE при удалении столбцов с пропусками:')
print(score_dataset(reduced_X_train, reduced_X_test, y_train, y_test))

In [ ]:
from sklearn.impute import SimpleImputer

my_imputer = SimpleImputer()  # по умолчанию заполняет средним
imputed_X_train = pd.DataFrame(my_imputer.fit_transform(X_train))
imputed_X_test = pd.DataFrame(my_imputer.transform(X_test))

imputed_X_train.columns = X_train.columns
imputed_X_test.columns = X_test.columns

print('MAE при заполнении пропусков средним:')
print(score_dataset(imputed_X_train, imputed_X_test, y_train, y_test))

In [ ]:
X_train_plus = X_train.copy()
X_test_plus = X_test.copy()

for col in cols_with_missing:
    X_train_plus[col + '_was_missing'] = X_train_plus[col].isnull()
    X_test_plus[col + '_was_missing'] = X_test_plus[col].isnull()

X_train_plus.head()

In [ ]:
my_imputer = SimpleImputer()
imputed_X_train_plus = pd.DataFrame(my_imputer.fit_transform(X_train_plus))
imputed_X_test_plus = pd.DataFrame(my_imputer.transform(X_test_plus))

imputed_X_train_plus.columns = X_train_plus.columns
imputed_X_test_plus.columns = X_test_plus.columns

print('MAE при заполнении пропусков + флаг отсутствия:')
print(score_dataset(imputed_X_train_plus, imputed_X_test_plus, y_train, y_test))

In [ ]:
y = data['price']
X_full = data.drop(['price'], axis=1)

X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_full, y, train_size=0.8, test_size=0.2, random_state=0
)

# Удаляем столбцы с пропусками (числовые)
cols_with_missing = [col for col in X_train_full.columns if X_train_full[col].isnull().any()]
X_train_full.drop(cols_with_missing, axis=1, inplace=True)
X_test_full.drop(cols_with_missing, axis=1, inplace=True)

# Выбираем категориальные (низкая кардинальность) и числовые признаки
low_cardinality_cols = [
    cname for cname in X_train_full.columns
    if X_train_full[cname].nunique() < 15 and X_train_full[cname].dtype == 'object'
]
numerical_cols = [
    cname for cname in X_train_full.columns
    if X_train_full[cname].dtype in ['int64', 'float64']
]

print('Категориальные признаки:', low_cardinality_cols)
print('Числовые признаки:', numerical_cols)

my_cols = low_cardinality_cols + numerical_cols
X_train = X_train_full[my_cols].copy()
X_test = X_test_full[my_cols].copy()

X_train.head()

In [ ]:
drop_X_train = X_train.select_dtypes(exclude=['object'])
drop_X_test = X_test.select_dtypes(exclude=['object'])

print('MAE без категориальных признаков:')
print(score_dataset(drop_X_train, drop_X_test, y_train, y_test))

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

object_cols = [cname for cname in X_train.columns if X_train[cname].dtype == 'object']

label_X_train = X_train.copy()
label_X_test = X_test.copy()

ordinal_encoder = OrdinalEncoder()
label_X_train[object_cols] = ordinal_encoder.fit_transform(label_X_train[object_cols])
label_X_test[object_cols] = ordinal_encoder.transform(label_X_test[object_cols])

print('MAE с Ordinal Encoding:')
print(score_dataset(label_X_train, label_X_test, y_train, y_test))

In [ ]:
from sklearn.preprocessing import OneHotEncoder

OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
OH_cols_train = pd.DataFrame(OH_encoder.fit_transform(X_train[object_cols]))
OH_cols_test = pd.DataFrame(OH_encoder.transform(X_test[object_cols]))

OH_cols_train.index = X_train.index
OH_cols_test.index = X_test.index

num_X_train = X_train.drop(object_cols, axis=1)
num_X_test = X_test.drop(object_cols, axis=1)

OH_X_train = pd.concat([num_X_train, OH_cols_train], axis=1)
OH_X_test = pd.concat([num_X_test, OH_cols_test], axis=1)

OH_X_train.columns = OH_X_train.columns.astype(str)
OH_X_test.columns = OH_X_test.columns.astype(str)

print('MAE с One-Hot Encoding:')
print(score_dataset(OH_X_train, OH_X_test, y_train, y_test))